# DS483 SMS spam detection
Executed research implementation for Tasks 1–5. Data: Almeida and Hidalgo (2011), UCI SMS Spam Collection, CC BY 4.0. No synthetic observations. Positive class is spam. All models share the same deduplicated 70/15/15 split. Group members must review and independently explain this assisted work; this file does not establish their authorship or development history.

Run all cells in order with Python 3.11+ and the packages in requirements.txt. First run needs internet for UCI data and the Google compact BERT checkpoint. CPU execution is supported. Outputs go to results/.

In [1]:
import os, re, json, time, random, copy, hashlib, urllib.request, zipfile, io, platform
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, average_precision_score, confusion_matrix, classification_report
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from nltk.stem import PorterStemmer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
SEED=483
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(4)
torch.use_deterministic_algorithms(True)
OUT=Path('results'); OUT.mkdir(exist_ok=True)
print('Python',platform.python_version(),'PyTorch',torch.__version__,'CPU threads',torch.get_num_threads())

Python 3.12.14 PyTorch 2.14.1+cpu CPU threads 4


## Task 1 Text preprocessing and exploration

In [2]:
url='https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'
datafile=Path('data/SMSSpamCollection')
if not datafile.exists():
    datafile.parent.mkdir(exist_ok=True)
    z=zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url,timeout=60).read()))
    datafile.write_bytes(z.read('SMSSpamCollection'))
raw=datafile.read_bytes()
rows=[line.split('\t',1) for line in raw.decode('utf-8').splitlines() if line.strip()]
df=pd.DataFrame(rows,columns=['label','text'])
assert df.label.isin(['ham','spam']).all() and df.text.str.strip().ne('').all()
df['id']=np.arange(len(df)); df['y']=(df.label=='spam').astype(int)
df['chars']=df.text.str.len(); df['words']=df.text.str.split().str.len()
df['digits']=df.text.str.count(r'\d');df['exclamations']=df.text.str.count('!')
df['url']=df.text.str.contains(r'https?://|www\.',case=False,regex=True).astype(int)
print('File SHA256:',hashlib.sha256(raw).hexdigest())
print('Raw rows',len(df));print(df.label.value_counts().to_string())
print(df.groupby('label')[['chars','words','digits','exclamations','url']].mean().round(3).to_string())
# Normalize whitespace/case for duplicate grouping, before splitting.
df['key']=df.text.str.lower().str.replace(r'\s+',' ',regex=True).str.strip()
conflicts=df.groupby('key').y.nunique(); assert conflicts.max()==1
unique=df.drop_duplicates('key').copy().reset_index(drop=True)
train,held=train_test_split(unique,test_size=.30,random_state=SEED,stratify=unique.y)
val,test=train_test_split(held,test_size=.50,random_state=SEED,stratify=held.y)
assert not(set(train.key)&set(test.key) or set(train.key)&set(val.key) or set(val.key)&set(test.key))
for name,frame in [('train',train),('validation',val),('test',test)]:
    frame[['id','label','text']].to_csv(OUT/(name+'.csv'),index=False)
    print(name,len(frame),frame.label.value_counts().to_dict())
# Retain negation and short SMS words; do not impose a generic stopword list.
def clean(text):
    text=re.sub(r'https?://\S+|www\.\S+',' urltoken ',text.lower())
    text=re.sub(r'[£$€]',' moneytoken ',text)
    text=re.sub(r'\d+',' numbertoken ',text)
    return ' '.join(re.findall(r"[a-z]+(?:'[a-z]+)?",text))
stemmer=PorterStemmer()
def stemmed(text):return ' '.join(stemmer.stem(t) for t in clean(text).split())
examples=df.groupby('label',sort=True).head(2)[['label','text']].copy()
examples['clean']=examples.text.map(clean);examples['stemmed_alternative']=examples.text.map(stemmed)
print(examples.to_string(index=False))
eda={'raw_rows':len(df),'counts':df.label.value_counts().to_dict(),'unique_rows':len(unique),'removed_duplicates':len(df)-len(unique),'sha256':hashlib.sha256(raw).hexdigest(),'means':df.groupby('label')[['chars','words','digits','exclamations','url']].mean().round(3).to_dict(),'splits':{k:{'n':len(v),'counts':v.label.value_counts().to_dict()} for k,v in [('train',train),('validation',val),('test',test)]}}
examples.to_json(OUT/'preprocessing.json',orient='records',indent=2)

File SHA256: 7d039a24a6083ed9ef0f806ebad56bbb976e3aeb8de05669173bfdc4996c239d
Raw rows 5574
label
ham     4827
spam     747
         chars   words  digits  exclamations    url
label                                              
ham     71.472  14.304   0.305         0.177  0.000
spam   138.676  23.912  15.813         0.730  0.142
train 3611 {'ham': 3162, 'spam': 449}
validation 774 {'ham': 677, 'spam': 97}
test 774 {'ham': 678, 'spam': 96}
label                                                                                                                                                        text                                                                                                                                                                                                  clean                                                                                                                                                                                 stemmed_alternative

Regular expressions normalize variable URLs, currencies and digit runs while retaining their presence. Tokenization removes punctuation from the classical representation; punctuation counts remain available for descriptive analysis only. Stopwords and negation stay because short messages contain little context. Porter stemming is demonstrated but is not used by default because SMS abbreviations can be distorted. The neural models retain sequence order; BERT receives original text and its pretrained tokenizer rather than stemmed tokens. These choices are assessed without fitting any vocabulary on validation or test messages.

## Task 2 TF IDF and cosine similarity

In [3]:
vectorizer=TfidfVectorizer(preprocessor=clean,token_pattern=r"(?u)\b\w+\b",ngram_range=(1,2),min_df=2,sublinear_tf=True)
Xtr=vectorizer.fit_transform(train.text);Xv=vectorizer.transform(val.text);Xt=vectorizer.transform(test.text)
print('Training vocabulary',len(vectorizer.vocabulary_),'shape',Xtr.shape)
# Retrieve training neighbours for one held-out example of each class.
similar=[]
for label in ['ham','spam']:
    q=val[val.label==label].iloc[0]
    scores=cosine_similarity(vectorizer.transform([q.text]),Xtr).ravel()
    for rank,j in enumerate(np.argsort(-scores)[:3],1):
        similar.append({'query_id':int(q.id),'query_label':label,'query':q.text,'rank':rank,'neighbor_id':int(train.iloc[j].id),'neighbor_label':train.iloc[j].label,'neighbor':train.iloc[j].text,'cosine':float(scores[j])})
print(pd.DataFrame(similar).to_string(index=False))
Path(OUT/'similarity.json').write_text(json.dumps(similar,indent=2))

Training vocabulary 9453 shape (3611, 9453)
 query_id query_label                                                                                                                                                                        query  rank  neighbor_id neighbor_label                                                                                                                                                 neighbor   cosine
     1491         ham Cant believe i said so many things to you this morning when all i really wanted to say was good morning, i love you! Have a beautiful morning. See you in the library later.     1         3612            ham                                                                                                                         No. I.ll meet you in the library 0.265121
     1491         ham Cant believe i said so many things to you this morning when all i really wanted to say was good morning, i love you! Have a beautiful morning. See you i

With sublinear term frequency, tf = 1 + log(count) for a present term. Smoothed idf = log((1+n)/(1+df)) + 1, where n and df refer only to training messages. TF–IDF products are L2-normalized. Cosine equals the dot product divided by the vector norms, and therefore becomes a dot product for these vectors. Rare shared words and bigrams contribute more than common words. High lexical similarity is useful for repeated campaigns but does not guarantee equivalent meaning. Normalization can also make unrelated phone numbers look alike.

## Task 3 Classical classifiers and common evaluation

In [4]:
ytr=train.y.to_numpy(); yv=val.y.to_numpy(); yt=test.y.to_numpy()
results=[]; probabilities={}; fits={}
def evaluate(name,p,seconds,complexity):
    pred=(p>=.5).astype(int)
    pr,rc,f1,_=precision_recall_fscore_support(yt,pred,average='binary',zero_division=0)
    macro=precision_recall_fscore_support(yt,pred,average='macro',zero_division=0)[2]
    tn,fp,fn,tp=confusion_matrix(yt,pred,labels=[0,1]).ravel()
    row=dict(model=name,accuracy=accuracy_score(yt,pred),spam_precision=pr,spam_recall=rc,spam_f1=f1,macro_f1=macro,average_precision=average_precision_score(yt,p),tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp),training_seconds=seconds,complexity=int(complexity))
    results.append(row); probabilities[name]=p
    print(name);print(classification_report(yt,pred,target_names=['ham','spam'],digits=4))
    print('Confusion [[TN,FP],[FN,TP]]',[[int(tn),int(fp)],[int(fn),int(tp)]])
    return row
for name,options in [('Naive Bayes',[.1,.5,1.0]),('Logistic Regression',[1.,4.,10.])]:
    best=None;t0=time.perf_counter()
    for value in options:
        model=MultinomialNB(alpha=value) if name=='Naive Bayes' else LogisticRegression(C=value,max_iter=1000,random_state=SEED)
        model.fit(Xtr,ytr)
        vf=precision_recall_fscore_support(yv,model.predict(Xv),average='binary',zero_division=0)[2]
        if best is None or vf>best[0]:best=(vf,value,model)
    model=best[2];fits[name]=model
    print('Selected',name,'parameter',best[1],'validation spam F1',best[0])
    evaluate(name,model.predict_proba(Xt)[:,1],time.perf_counter()-t0,(model.feature_log_prob_.size+model.class_log_prior_.size) if name=='Naive Bayes' else (model.coef_.size+model.intercept_.size))
# Explain linear associations, not causal effects.
features=vectorizer.get_feature_names_out();coef=fits['Logistic Regression'].coef_[0]
print('Largest spam coefficients:',list(zip(features[np.argsort(-coef)[:12]],np.round(np.sort(coef)[::-1][:12],3))))

Selected Naive Bayes parameter 0.1 validation spam F1 0.9405405405405406
Naive Bayes
              precision    recall  f1-score   support

         ham     0.9854    0.9985    0.9919       678
        spam     0.9885    0.8958    0.9399        96

    accuracy                         0.9858       774
   macro avg     0.9870    0.9472    0.9659       774
weighted avg     0.9858    0.9858    0.9855       774

Confusion [[TN,FP],[FN,TP]] [[677, 1], [10, 86]]
Selected Logistic Regression parameter 10.0 validation spam F1 0.9340659340659341
Logistic Regression
              precision    recall  f1-score   support

         ham     0.9868    0.9956    0.9912       678
        spam     0.9667    0.9062    0.9355        96

    accuracy                         0.9845       774
   macro avg     0.9768    0.9509    0.9633       774
weighted avg     0.9843    0.9845    0.9843       774

Confusion [[TN,FP],[FN,TP]] [[675, 3], [9, 87]]
Largest spam coefficients: [('numbertoken', np.float64(10.689)

## Task 4 LSTM sequence classifier

In [5]:
counter=Counter(t for s in train.text for t in clean(s).split())
vocab={'<PAD>':0,'<UNK>':1};vocab.update({t:i+2 for i,(t,c) in enumerate(counter.most_common(9998)) if c>=2})
MAXLEN=80
def encode(texts):
    a=np.zeros((len(texts),MAXLEN),dtype=np.int64);length=[]
    for i,s in enumerate(texts):
        ids=[vocab.get(t,1) for t in clean(s).split()][:MAXLEN] or [1]
        a[i,:len(ids)]=ids;length.append(len(ids))
    return torch.tensor(a),torch.tensor(length)
class SMSLSTM(nn.Module):
    def __init__(self):
        super().__init__();self.emb=nn.Embedding(len(vocab),64,padding_idx=0);self.rnn=nn.LSTM(64,64,batch_first=True);self.drop=nn.Dropout(.25);self.fc=nn.Linear(64,1)
    def forward(self,x,length):
        packed=nn.utils.rnn.pack_padded_sequence(self.emb(x),length.cpu(),batch_first=True,enforce_sorted=False)
        _,(h,_)=self.rnn(packed);return self.fc(self.drop(h[-1])).squeeze(-1)
ax,al=encode(train.text.tolist());vx,vl=encode(val.text.tolist());tx,tl=encode(test.text.tolist())
loader=DataLoader(TensorDataset(ax,al,torch.tensor(ytr,dtype=torch.float32)),batch_size=64,shuffle=True,generator=torch.Generator().manual_seed(SEED))
lstm=SMSLSTM();optimizer=torch.optim.Adam(lstm.parameters(),lr=.002)
lossfn=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float((ytr==0).sum()/(ytr==1).sum())))
bestf=-1;history=[];t0=time.perf_counter()
for epoch in range(1,11):
    lstm.train();total=0
    for x,l,y in loader:
        optimizer.zero_grad();loss=lossfn(lstm(x,l),y);loss.backward();nn.utils.clip_grad_norm_(lstm.parameters(),1);optimizer.step();total+=loss.item()*len(y)
    lstm.eval()
    with torch.no_grad():vp=torch.sigmoid(lstm(vx,vl)).numpy()
    f=precision_recall_fscore_support(yv,vp>=.5,average='binary',zero_division=0)[2]
    history.append({'epoch':epoch,'train_loss':total/len(train),'validation_f1':f})
    print(history[-1])
    if f>bestf:bestf=f;beststate=copy.deepcopy(lstm.state_dict());bestepoch=epoch
lstm.load_state_dict(beststate);lstm.eval()
with torch.no_grad():lp=torch.sigmoid(lstm(tx,tl)).numpy()
evaluate('LSTM',lp,time.perf_counter()-t0,sum(p.numel() for p in lstm.parameters()))
print('Selected epoch',bestepoch,'vocabulary',len(vocab),'max length',MAXLEN)
# Diagnostic only: reverse test token order without changing the trained model.
reverse=tx.clone()
for i,l in enumerate(tl):reverse[i,:l]=tx[i,:l].flip(0)
with torch.no_grad():rp=torch.sigmoid(lstm(reverse,tl)).numpy()
reverse_f1=precision_recall_fscore_support(yt,rp>=.5,average='binary',zero_division=0)[2]
print('Reversed token order diagnostic F1',reverse_f1)
print('Fraction of test sequences truncated',np.mean([len(clean(s).split())>MAXLEN for s in test.text]))

{'epoch': 1, 'train_loss': 0.7302099819230888, 'validation_f1': 0.776255707762557}
{'epoch': 2, 'train_loss': 0.360456296421507, 'validation_f1': 0.8736842105263158}
{'epoch': 3, 'train_loss': 0.19546529004462238, 'validation_f1': 0.9035532994923858}
{'epoch': 4, 'train_loss': 0.11709386019468951, 'validation_f1': 0.9278350515463918}
{'epoch': 5, 'train_loss': 0.07632356742048768, 'validation_f1': 0.9333333333333333}
{'epoch': 6, 'train_loss': 0.042265601575063046, 'validation_f1': 0.9214659685863874}
{'epoch': 7, 'train_loss': 0.024963400011757873, 'validation_f1': 0.9230769230769231}
{'epoch': 8, 'train_loss': 0.01503998756870226, 'validation_f1': 0.9263157894736842}
{'epoch': 9, 'train_loss': 0.006991851261985336, 'validation_f1': 0.9148936170212766}
{'epoch': 10, 'train_loss': 0.005721215431121178, 'validation_f1': 0.9157894736842105}
LSTM
              precision    recall  f1-score   support

         ham     0.9895    0.9735    0.9814       678
        spam     0.8318    0.9271  

## Task 5 Fine tune pretrained compact BERT

In [6]:
MODEL='google/bert_uncased_L-2_H-128_A-2'
# This is pretrained BERT Tiny: two transformer layers, 128 hidden units, two heads.
# The classification head is newly initialized; every encoder parameter is fine-tuned.
tokenizer=AutoTokenizer.from_pretrained(MODEL,revision='30b0a37ccaaa32f332884b96992754e246e48c5f')
bert=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=2,revision='30b0a37ccaaa32f332884b96992754e246e48c5f')
def bertdata(frame):
    enc=tokenizer(frame.text.tolist(),padding='max_length',truncation=True,max_length=96,return_tensors='pt')
    return TensorDataset(enc['input_ids'],enc['attention_mask'],torch.tensor(frame.y.to_numpy(),dtype=torch.long))
btr,bv,bt=bertdata(train),bertdata(val),bertdata(test)
bloader=DataLoader(btr,batch_size=32,shuffle=True,generator=torch.Generator().manual_seed(SEED))
def bertprobs(ds):
    bert.eval();out=[]
    with torch.no_grad():
        for ids,mask,y in DataLoader(ds,batch_size=64):out.extend(torch.softmax(bert(input_ids=ids,attention_mask=mask).logits,dim=1)[:,1].tolist())
    return np.array(out)
optimizer=torch.optim.AdamW(bert.parameters(),lr=5e-5,weight_decay=.01)
bloss=nn.CrossEntropyLoss(weight=torch.tensor([1.,float((ytr==0).sum()/(ytr==1).sum())]))
bestf=-1;bhistory=[];t0=time.perf_counter()
for epoch in range(1,5):
    bert.train();total=0
    for ids,mask,y in bloader:
        optimizer.zero_grad();logits=bert(input_ids=ids,attention_mask=mask).logits;loss=bloss(logits,y);loss.backward();nn.utils.clip_grad_norm_(bert.parameters(),1.);optimizer.step();total+=loss.item()*len(y)
    vp=bertprobs(bv);f=precision_recall_fscore_support(yv,vp>=.5,average='binary',zero_division=0)[2]
    bhistory.append({'epoch':epoch,'train_loss':total/len(train),'validation_f1':f});print(bhistory[-1])
    if f>bestf:bestf=f;bs=copy.deepcopy(bert.state_dict());bepoch=epoch
bert.load_state_dict(bs)
evaluate('BERT Tiny',bertprobs(bt),time.perf_counter()-t0,sum(p.numel() for p in bert.parameters()))
print('Selected epoch',bepoch,'checkpoint revision',getattr(bert.config,'_commit_hash',None))

{'epoch': 1, 'train_loss': 0.39225079552911385, 'validation_f1': 0.9119170984455959}
{'epoch': 2, 'train_loss': 0.14331442494140342, 'validation_f1': 0.9354838709677419}
{'epoch': 3, 'train_loss': 0.10821542811557563, 'validation_f1': 0.9405405405405406}
{'epoch': 4, 'train_loss': 0.08921473145916155, 'validation_f1': 0.9347826086956522}
BERT Tiny
              precision    recall  f1-score   support

         ham     0.9840    0.9985    0.9912       678
        spam     0.9884    0.8854    0.9341        96

    accuracy                         0.9845       774
   macro avg     0.9862    0.9420    0.9626       774
weighted avg     0.9846    0.9845    0.9841       774

Confusion [[TN,FP],[FN,TP]] [[677, 1], [11, 85]]
Selected epoch 3 checkpoint revision 30b0a37ccaaa32f332884b96992754e246e48c5f


## Consolidated results and actual errors

In [7]:
summary=pd.DataFrame(results);print(summary.round(5).to_string(index=False))
errors=[]
for name,p in probabilities.items():
    pred=(p>=.5).astype(int)
    for i in np.flatnonzero(pred!=yt):
        errors.append({'model':name,'id':int(test.iloc[i].id),'truth':test.iloc[i].label,'prediction':'spam' if pred[i] else 'ham','spam_probability':float(p[i]),'text':test.iloc[i].text})
errorsdf=pd.DataFrame(errors);errorsdf.to_csv(OUT/'errors.csv',index=False)
print(errorsdf.groupby(['model','truth']).head(2).to_string(index=False))
predictions=test[['id','label','text']].copy()
for name,p in probabilities.items():predictions[name]=p
predictions.to_csv(OUT/'predictions.csv',index=False)
summary.to_csv(OUT/'metrics.csv',index=False)
# Test is never used to fit, tune, stop training or alter a decision threshold.
# Small diagnostic slices are not demographic fairness measurements.
slices=[]
for name,p in probabilities.items():
    for title,mask in [('short <=40 chars',test.chars.to_numpy()<=40),('long >40 chars',test.chars.to_numpy()>40),('contains digit',test.digits.to_numpy()>0),('no digit',test.digits.to_numpy()==0)]:
        if mask.sum():slices.append({'model':name,'slice':title,'n':int(mask.sum()),'spam_support':int(yt[mask].sum()),'accuracy':float(accuracy_score(yt[mask],p[mask]>=.5))})
print(pd.DataFrame(slices).round(4).to_string(index=False))
metadata={'eda':eda,'results':results,'lstm_history':history,'bert_history':bhistory,'lstm_selected_epoch':bestepoch,'bert_selected_epoch':bepoch,'reverse_f1':reverse_f1,'vocabulary':len(vectorizer.vocabulary_),'lstm_vocabulary':len(vocab),'bert_revision':getattr(bert.config,'_commit_hash',None),'slices':slices,'versions':{'python':platform.python_version(),'torch':torch.__version__}}
(OUT/'run_summary.json').write_text(json.dumps(metadata,indent=2))
assert all(r['tn']+r['fp']+r['fn']+r['tp']==len(test) for r in results)
assert len(probabilities)==4 and all(len(p)==len(test) for p in probabilities.values())
print('VERIFIED: all four classifiers evaluated on identical test IDs; no split overlap.')

              model  accuracy  spam_precision  spam_recall  spam_f1  macro_f1  average_precision  tn  fp  fn  tp  training_seconds  complexity
        Naive Bayes   0.98579         0.98851      0.89583  0.93989   0.96592            0.96749 677   1  10  86           0.00692       18908
Logistic Regression   0.98450         0.96667      0.90625  0.93548   0.96334            0.97613 675   3   9  87           0.07033        9454
               LSTM   0.96770         0.83178      0.92708  0.87685   0.92913            0.95673 660  18   7  89          10.07087      222401
          BERT Tiny   0.98450         0.98837      0.88542  0.93407   0.96264            0.97537 677   1  11  85          54.33166     4386178
              model   id truth prediction  spam_probability                                                                                                                                                     text
        Naive Bayes 4514  spam        ham          0.088093             

## Interpretation and scope
Read the accompanying report for the completed comparison and deployment judgment. Accuracy alone is insufficient because ham dominates. Spam recall describes missed spam; spam precision describes the reliability of alerts; ham false positives are explicitly counted. Average precision assesses ranking across thresholds. One fixed split and seed do not establish statistical superiority or production performance. No protected demographic labels exist, so demographic fairness cannot be measured. The token reversal diagnostic changes the input distribution and does not isolate a causal benefit of order. The corpus is old and cannot establish performance on contemporary multilingual or adversarial traffic.

## Sources consulted
Almeida, T., & Hidalgo, J. (2011). SMS Spam Collection [Data set]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CC84

Google. (n.d.). BERT Miniatures model card. https://huggingface.co/google/bert_uncased_L-2_H-128_A-2

Scikit-learn developers. (n.d.). Feature extraction and model evaluation documentation. https://scikit-learn.org/stable/modules/feature_extraction.html and https://scikit-learn.org/stable/modules/model_evaluation.html

PyTorch contributors. (n.d.). LSTM documentation. https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html